# Chapter 16: Model Selection. Computing exercises

These exercises are from *Bayes, Artificial Intelligence, and Deep Learning: Foundations of Data Science* by Nick Polson and Vadim Sokolov, <https://github.com/VadimSokolov/bayes_ai_deep_learning>. The text is licensed under CC BY 4.0 and the code under the MIT License.

The code of the chapter, which several exercises refer to, is in [`code/16-select.R`](https://github.com/VadimSokolov/bayes_ai_deep_learning/blob/main/code/16-select.R).

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/VadimSokolov/bayes_ai_deep_learning/blob/main/exercises/16-select.ipynb)

In [ ]:
# Packages that the exercises of this notebook use; the ones that are missing are installed
pkgs <- c("gamlr", "MASS", "randomForest")
new <- setdiff(pkgs, rownames(installed.packages()))
if (length(new) > 0) install.packages(new, repos = "https://cloud.r-project.org")

# The exercises read files from data/: download the datasets of the book once
if (!dir.exists("data")) {
  options(timeout = max(3600, getOption("timeout")))
  download.file("https://github.com/VadimSokolov/bayes_ai_deep_learning/releases/latest/download/data.zip", "data.zip", mode = "wb")
  unzip("data.zip")
  unlink("data.zip")
}

## Exercise 16.5: Bias-Variance Tradeoff: Polynomial Regression

Use the `Boston` housing dataset from the `MASS` package to explore the bias-variance tradeoff.

a) Split the data into 80% training and 20% test. Fit polynomial models of degree 1 through 12 predicting `medv` from `lstat`. Compute training and test MSE for each degree.

b) Plot training MSE and test MSE versus polynomial degree on the same graph. Identify the optimal degree.

c) Repeat the experiment 50 times with different random splits. Plot the average test MSE with error bars (standard deviation). Does the optimal degree change?

## Exercise 16.6: Residual Diagnostics for Polynomial Regression

Example 16.1 used Q-Q plots and residual/scatter plots to check whether a model's assumed error distribution is reasonable. The prediction interval stated at the top of this chapter,
$$
\hat{y} \pm t_{n-2,\,1-\alpha/2}\, s\sqrt{1+\frac{1}{n}+\frac{(x-\bar{x})^2}{\sum_{i=1}^n(x_i-\bar{x})^2}},
$$
is only trustworthy when the residuals are approximately normal with constant variance $\mathrm{Var}\left(\epsilon_i\right) = \sigma^2$. Return to the polynomial fits of `medv` on `lstat` in `MASS::Boston` from the bias-variance exercise and check whether those assumptions actually hold. No train/test split is needed here: diagnostics concern the fitted model itself, not out-of-sample error.

a) Fit polynomial regressions of degree 1, 5, and 12 to the full `Boston` data, predicting `medv` from `lstat`. Report $R^2$ and adjusted $R^2$ for each. How much of the improvement in fit does degree 5 already capture?

b) For each of the three fits, produce (i) a scatter plot of residuals against fitted values with a horizontal reference line at 0 and (ii) a normal Q-Q plot of the residuals (`qqnorm`/`qqline`, exactly as in Example 16.1). For the degree-5 fit, also produce the full base-R diagnostic set (`plot(fit)`), which adds the scale-location and residuals-versus-leverage panels.

c) Run `shapiro.test` on each set of residuals to test normality. Separately, test for non-constant variance by regressing the squared residuals on the fitted values, `lm(resid(fit)^2 ~ fitted(fit))`, and checking whether the slope differs significantly from 0.

d) Identify the handful of observations that produce the largest residuals in all three models (hint: inspect `table(Boston$medv)`). Explain why raising the degree from 5 to 12 barely changes the diagnostic picture, even though degree 5 minimized averaged out-of-sample MSE in the bias-variance exercise. What does this imply about trusting the prediction interval above for this dataset, at any polynomial degree?

## Exercise 16.7: Cross-Validation from Scratch

Implement k-fold cross-validation from scratch (without using a library function).

a) Write a function `kfold_cv(X, y, k, degree)` that splits data into $k$ folds, fits a polynomial of the given degree on $k-1$ folds, predicts on the held-out fold, and returns the average MSE across all folds.

b) Apply your function to the `Boston` data (predicting `medv` from `lstat`) with $k = 5$ and degrees 1 through 8. Which degree minimizes CV error?

c) Compare $k = 5$ and $k = 10$ cross-validation. Are the selected models the same?

## Exercise 16.8: Wine Clustering

Use `wine.csv` containing 6,497 wines with 11 chemical properties, color (red/white), and quality ratings.

a) Scale the 11 chemical features and perform k-means clustering with K=2. What do the clusters correspond to?

b) Use AIC/BIC to choose K. Try K=1 to 50 and plot the information criteria.

c) With your chosen K, examine cluster quality distributions. Do clusters help identify high-quality wines?

d) Compare: regressing quality on cluster membership vs. regressing on raw features. Which gives better OOS R²?

## Exercise 16.9: Explaining a Black-Box Classifier: Importance, PDP, and Shapley

Use the German credit data (`credit.csv`) to fit a random forest classifier predicting `Default` from `duration`, `amount`, `installment`, `age`, and `history` (recode `history` into `good`, `poor`, `terrible`, as in earlier chapters). Dedicated explanation packages such as `fastshap`, `iml`, or `DALEX` may not be installed in every environment; check with `requireNamespace()` and, if they are unavailable, implement the following by hand.

a) Split the data 70/30 into training and test sets and fit a random forest. Compute the baseline test accuracy. Compute permutation variable importance for each of the five features: shuffle the feature's values in the test set, re-predict, and record the drop in accuracy relative to the baseline. Average over several shuffles per feature and rank the features.

b) For the feature with the largest permutation importance, construct a partial dependence plot: sweep that feature over a grid of values, overwrite it in the training data at each grid point, and average the model's predicted probability of default over all other (unchanged) rows.

c) Pick the test-set instance with the highest predicted default probability. With the five features $F = \{$`duration`, `amount`, `installment`, `age`, `history`$\}$, compute the exact Shapley value of each feature for this instance,
$$
\phi_i = \sum_{S \subseteq F \setminus \{i\}} \frac{|S|!\,(|F|-|S|-1)!}{|F|!}\Big[v(S \cup \{i\}) - v(S)\Big],
$$
where the value function $v(S) = \mathrm{E}_{X_{-S}}\left(f(x_S, X_{-S})\right)$ is the model's average predicted probability over a background sample with the features in $S$ fixed at the instance's values and the remaining features left at their background values. Since $|F|=5$, all $2^5 = 32$ subsets can be enumerated exactly. Verify the efficiency property $\sum_i \phi_i = f(x) - v(\emptyset)$.

d) Compute the empirical default rate for each level of `history`. Reconcile this with the sign of the Shapley contribution of `history` found in (c), and comment on what this reveals about interpreting black-box explanations fit to real data (compare with the "wolf detector" story earlier in the chapter).

## Exercise 16.10: Evaluating Classifiers: ROC/AUC, Calibration, and Proper Scores

Use the same German credit data and predictors as Exercise 16.9 (`duration`, `amount`, `installment`, `age`, recoded `history`) to fit a logistic regression predicting `Default`, using a 70/30 train/test split. Packages such as `pROC` and `ROCR` may not be installed; check with `requireNamespace()` and compute the following by hand if necessary.

a) Compute the ROC curve on the test set by sweeping over all thresholds implied by the fitted probabilities, and the AUC two ways: (i) the trapezoidal area under the ROC curve, and (ii) the rank-sum (Mann-Whitney) identity
$$
\text{AUC} = \frac{\sum_{i:\,y_i=1} r_i - P(P+1)/2}{P N},
$$
where $r_i$ is the rank of $\hat p_i$ among all test-set predictions, and $P$, $N$ are the numbers of positives (defaults) and negatives. Confirm the two methods agree, and plot the ROC curve.

b) Construct a calibration (reliability) plot: bin the test-set predictions into deciles, and for each bin plot the mean predicted probability against the observed default frequency, together with the 45-degree reference line.

c) Compute the Brier score and the log score on the test set,
$$
\text{Brier} = \frac{1}{n}\sum_{i=1}^n (\hat p_i - y_i)^2, \qquad
\text{LogScore} = -\frac{1}{n}\sum_{i=1}^n\Big[y_i \log \hat p_i + (1-y_i)\log(1-\hat p_i)\Big].
$$

d) Apply the strictly increasing transform $\tilde p = \hat p^3 / \big(\hat p^3 + (1-\hat p)^3\big)$ to the fitted probabilities. This exaggerates confidence (pushes probabilities toward 0 or 1) but cannot change the ranking of cases. Recompute the AUC, calibration table, Brier score, and log score for $\tilde p$, and overlay the new calibration curve on the plot from (b). Explain why the AUC is unchanged while the other three diagnostics indicate worse calibration, and what this implies about relying on AUC alone to evaluate a probabilistic classifier.